In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 *  NORMALIZATION: Protecting Truth, Eliminating Anomalies
 * ─────────────────────────────────────────────────────────────────────
*/

In [ ]:
/**
 * UNDERSTANDING 1NF, 2NF, 3NF — The Foundation of Relational Database Design
 * ─────────────────────────────────────────────────────────────────────
 *
 * TABLE OF CONTENTS:
 * 1. WHAT IS NORMALIZATION? ............. Each fact stored once
 * 2. UPDATE ANOMALY .................... Duplicated data → inconsistent updates
 * 3. INSERT ANOMALY .................... Duplicated data → complex inserts
 * 4. DELETE ANOMALY .................... Unclear relationships → data loss
 * 5. FIRST NORMAL FORM (1NF) ........... Atomic values, no repeating groups
 * 6. SECOND NORMAL FORM (2NF) .......... No partial dependencies
 * 7. THIRD NORMAL FORM (3NF) ........... No transitive dependencies
 * ─────────────────────────────────────────────────────────────────────
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * SAMPLE TABLES — For illustrating normalization concepts
 * ─────────────────────────────────────────────────────────────────────
 *
 * Bad design: denormalized_enrollments (student_id, student_name, course_id, 
 *              course_name, instructor_name, enrollment_date, grade)
 * 
 * Good design: students, courses, instructors, enrollments (normalized)
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 1. WHAT IS NORMALIZATION? — Each fact stored once
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ NORMALIZATION is the process of designing database tables so that  │
 * │ each real-world fact is stored EXACTLY ONCE, and everything else   │
 * │ references it using keys.                                          │
 * │                                                                    │
 * │ That single sentence contains the entire philosophy:               │
 * │   → Store the truth ONCE                                           │
 * │   → Avoid duplication                                              │
 * │   → Link using relationships                                       │
 * │                                                                    │
 * │ WHY NORMALIZATION MATTERS:                                         │
 * │   When you design from this mindset:                               │
 * │     → Schema becomes STABLE                                        │
 * │     → Data remains CONSISTENT                                      │
 * │     → Anomalies are PREVENTED                                      │
 * │                                                                    │
 * │ When you ignore it:                                                │
 * │     → Update anomalies (update some rows, miss others)             │
 * │     → Insert anomalies (need redundant data)                       │
 * │     → Delete anomalies (accidental data loss)                      │
 * │                                                                    │
 * │ Normalization is not about memorizing definitions.                 │
 * │ It is about PROTECTING TRUTH.                                      │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 2. UPDATE ANOMALY — Duplicated data → inconsistent updates
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ UPDATE ANOMALY happens when duplicated data gets updated           │
 * │ inconsistently.                                                    │
 * │                                                                    │
 * │ BAD DESIGN (Denormalized enrollments):                             │
 * │                                                                    │
 * │   enrollments table                                                │
 * │   ┌────────────┬───────────────┬───────────┬─────────────────┐     │
 * │   │ student_id │ student_name  │ course_id │ enrollment_date │     │
 * │   ├────────────┼───────────────┼───────────┼─────────────────┤     │
 * │   │ 1          │ Asha Sharma   │ 101       │ 2026-01-01      │     │
 * │   │ 1          │ Asha S.       │ 102       │ 2026-01-05      │     │
 * │   │ 2          │ Ravi Kumar    │ 101       │ 2026-01-02      │     │
 * │   └────────────┴───────────────┴───────────┴─────────────────┘     │
 * │                                                                    │
 * │ PROBLEM: Asha's name is stored TWICE:                              │
 * │   → Row 1: 'Asha Sharma'                                           │
 * │   → Row 2: 'Asha S.'                                               │
 * │                                                                    │
 * │ SCENARIO: Asha updates her name to 'Asha Sharma' officially.       │
 * │                                                                    │
 * │   UPDATE students SET name = 'Asha Sharma' WHERE id = 1;           │
 * │   (but students table doesn't exist in this design!)               │
 * │                                                                    │
 * │   Now what? You must update ALL enrollment rows:                   │
 * │     → UPDATE enrollments SET student_name = 'Asha Sharma'          │
 * │       WHERE student_id = 1;                                        │
 * │                                                                    │
 * │   If you miss one row:                                             │
 * │     → Inconsistent data                                            │
 * │     → Reports show two different names for same student            │
 * │     → Analytics break                                              │
 * │     → Joins produce contradictory results                          │
 * │                                                                    │
 * │ NORMALIZED DESIGN (No update anomaly):                             │
 * │                                                                    │
 * │   students table (one row per student):                            │
 * │   ┌────────────┬───────────────┐                                   │
 * │   │ student_id │ student_name  │                                   │
 * │   ├────────────┼───────────────┤                                   │
 * │   │ 1          │ Asha Sharma   │  ← one place                      │
 * │   └────────────┴───────────────┘                                   │
 * │                                                                    │
 * │   enrollments table (references by ID):                            │
 * │   ┌────────────┬───────────┬─────────────────┐                     │
 * │   │ student_id │ course_id │ enrollment_date │                     │
 * │   ├────────────┼───────────┼─────────────────┤                     │
 * │   │ 1          │ 101       │ 2026-01-01      │                     │
 * │   │ 1          │ 102       │ 2026-01-05      │                     │
 * │   └────────────┴───────────┴─────────────────┘                     │
 * │                                                                    │
 * │ Now name update:                                                   │
 * │   UPDATE students SET name = 'Asha Sharma' WHERE id = 1;           │
 * │   → ONE UPDATE                                                     │
 * │   → All enrollments automatically see the correct name via JOIN    │
 * │                                                                    │
 * │ UPDATE ANOMALY PREVENTED: Truth lives in ONE place.                │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 3. INSERT ANOMALY — Duplicated data → complex inserts
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ INSERT ANOMALY happens when you cannot insert data without         │
 * │ duplicating other data.                                            │
 * │                                                                    │
 * │ BAD DESIGN (Denormalized enrollments):                             │
 * │                                                                    │
 * │   enrollments table stores:                                        │
 * │     student_id, student_name, course_id, course_name,              │
 * │     instructor_name, enrollment_date                               │
 * │                                                                    │
 * │ PROBLEM: To insert a new enrollment:                               │
 * │   → Must repeat student_name (even though it's stored elsewhere)   │
 * │   → Must repeat course_name (even though it's defined once)        │
 * │   → Must repeat instructor_name (even though it's defined once)    │
 * │                                                                    │
 * │   INSERT INTO enrollments VALUES (                                 │
 * │     3, 'Neha Singh', 101, 'SQL Basics', 'Raj Kumar', ...           │ 
 * │   );                                                               │
 * │                                                                    │
 * │   If any of this repeated data is slightly wrong:                  │
 * │     → 'Neha Singh' vs 'Neha S.' → inconsistency                    │
 * │     → 'SQL Basics' vs 'SQL Fundamentals' → confusion               │
 * │                                                                    │
 * │   If a new column is added (e.g., student_phone):                  │
 * │     → Must update ALL insert statements                            │
 * │     → More complexity, more bugs                                   │
 * │                                                                    │
 * │ NORMALIZED DESIGN:                                                 │
 * │                                                                    │
 * │   Insert student ONCE:                                             │
 * │     INSERT INTO students (student_id, name) VALUES (3, 'Neha');    │
 * │                                                                    │
 * │   Insert enrollment ONCE:                                          │
 * │     INSERT INTO enrollments (student_id, course_id, enrolled_at)   │
 * │     VALUES (3, 101, NOW());                                        │
 * │                                                                    │
 * │   No duplication.                                                  │
 * │   Insertion is simple and error-free.                              │
 * │                                                                    │
 * │ INSERT ANOMALY PREVENTED: Each fact inserted once.                 │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 4. DELETE ANOMALY — Unclear relationships → data loss
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌─────────────────────────────────────────────────────────────────────┐
 * │ DELETE ANOMALY happens when deleting one piece of data              │
 * │ unintentionally removes other important data.                       │
 * │                                                                     │
 * │ BAD DESIGN (Denormalized enrollments):                              │
 * │                                                                     │
 * │   enrollments table stores EVERYTHING:                              │
 * │     student info, course info, enrollment info                      │
 * │                                                                     │
 * │ PROBLEM: A student withdraws from a course.                         │
 * │                                                                     │
 * │   DELETE FROM enrollments WHERE student_id = 1 AND course_id = 101; │
 * │                                                                     │
 * │   What happens?                                                     │
 * │     → Enrollment is gone (good)                                     │
 * │     → But also: Student information for that enrollment? Gone.      │
 * │     → Course information for that enrollment? Gone.                 │
 * │     → Payment record? If stored here, also gone.                    │
 * │                                                                     │
 * │   If student's only enrollment was deleted:                         │
 * │     → Student record disappears completely!                         │
 * │     → All history of that student lost                              │
 * │     → Financial audit impossible                                    │
 * │                                                                     │
 * │ NORMALIZED DESIGN (with proper foreign key rules):                  │
 * │                                                                     │
 * │   DELETE FROM enrollments WHERE student_id = 1 AND course_id = 101; │
 * │                                                                     │
 * │   → Only enrollment deleted                                         │
 * │   → Student still exists in students table                          │
 * │   → Course still exists in courses table                            │
 * │   → Payments still exist in payments table                          │
 * │                                                                     │
 * │   If we wanted to delete a student with enrollments:                │
 * │     → FOREIGN KEY with ON DELETE RESTRICT prevents accidental loss  │
 * │     → You must explicitly decide: CASCADE or RESTRICT               │
 * │                                                                     │
 * │ DELETE ANOMALY PREVENTED: Clear relationship rules.                 │
 * └─────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 5. FIRST NORMAL FORM (1NF) — Atomic values, no repeating groups
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌──────────────────────────────────────────────────────────────────────┐
 * │ A table is in FIRST NORMAL FORM (1NF) if:                            │
 * │                                                                      │
 * │   1. Every column holds a SINGLE, ATOMIC value                       │
 * │      → No lists, no comma-separated values                           │
 * │      → No arrays, no JSON blobs for structured data                  │
 * │                                                                      │
 * │   2. No repeating groups (like course1, course2, course3)            │
 * │                                                                      │
 * │   3. Each row represents ONE record of a single entity type          │
 * │                                                                      │
 * │ WHY ATOMIC VALUES MATTER:                                            │
 * │                                                                      │
 * │   BAD DESIGN (violates 1NF):                                         │
 * │   ┌────────────┬─────────────────┐                                   │
 * │   │ student_id │ course_ids      │                                   │
 * │   ├────────────┼─────────────────┤                                   │
 * │   │ 1          │ 2,4,6           │  ← list!                          │
 * │   └────────────┴─────────────────┘                                   │
 * │                                                                      │
 * │   To check if student 1 enrolled in course 4:                        │
 * │     → Must parse string '2,4,6'                                      │
 * │     → Split into array                                               │
 * │     → Search for '4'                                                 │
 * │     → Indexing useless                                               │
 * │     → Slow, complex                                                  │
 * │                                                                      │
 * │   CORRECT DESIGN (1NF compliant):                                    │
 * │   ┌────────────┬───────────┐                                         │
 * │   │ student_id │ course_id │                                         │
 * │   ├────────────┼───────────┤                                         │
 * │   │ 1          │ 2         │                                         │
 * │   │ 1          │ 4         │                                         │
 * │   │ 1          │ 6         │                                         │
 * │   └────────────┴───────────┘                                         │
 * │                                                                      │
 * │   Now:                                                               │
 * │     → Indexing works                                                 │
 * │     → Search becomes instant                                         │
 * │     → Joins are clean                                                │
 * │                                                                      │
 * │ REPEATING GROUPS PROBLEM:                                            │
 * │                                                                      │
 * │   BAD DESIGN (violates 1NF):                                         │
 * │   ┌────────────┬─────────────────┬─────────────────┬──────────────┐  │
 * │   │ student_id │ course1_notes   │ course2_notes   │ course3_notes│  │
 * │   ├────────────┼─────────────────┼─────────────────┼──────────────┤  │
 * │   │ 1          │ 'Learned JOINs' │ 'Learned INDEX' │ NULL         │  │
 * │   └────────────┴─────────────────┴─────────────────┴──────────────┘  │
 * │                                                                      │
 * │   Problems:                                                          │
 * │     → Assumes max 3 courses                                          │
 * │     → Schema must change if 4th course appears                       │
 * │     → Queries become complex                                         │
 * │                                                                      │
 * │   CORRECT DESIGN:                                                    │
 * │   ┌────────────┬───────────┬─────────────────────┐                   │
 * │   │ note_id    │ student_id│ note_text           │                   │
 * │   ├────────────┼───────────┼─────────────────────┤                   │
 * │   │ 1          │ 1         │ 'Learned JOINs'     │                   │
 * │   │ 2          │ 1         │ 'Learned INDEX'     │                   │
 * │   └────────────┴───────────┴─────────────────────┘                   │
 * │                                                                      │
 * │   Benefits:                                                          │
 * │     → Unlimited courses                                              │
 * │     → Schema stable                                                  │
 * │     → Queries clean                                                  │
 * │                                                                      │
 * │ 1NF IS THE BARE MINIMUM for serious databases.                       │
 * │ Without it: indexes fail, searches are slow, joins are unreliable.   │
 * └──────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 6. SECOND NORMAL FORM (2NF) — No partial dependencies
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ SECOND NORMAL FORM (2NF) applies ONLY to tables with               │
 * │ COMPOSITE PRIMARY KEYS.                                            │
 * │                                                                    │
 * │ A table is in 2NF if:                                              │
 * │   → It is already in 1NF                                           │
 * │   → Every non-key column depends on the ENTIRE composite key       │
 * │   → NO PARTIAL DEPENDENCIES                                        │
 * │                                                                    │
 * │ WHAT IS PARTIAL DEPENDENCY?                                        │
 * │                                                                    │
 * │   BAD DESIGN (violates 2NF):                                       │
 * │                                                                    │
 * │   enrollments table                                                │
 * │   PRIMARY KEY = (student_id, course_id)                            │
 * │                                                                    │
 * │   ┌────────────┬───────────┬─────────────────┬─────────────────┐   │
 * │   │ student_id │ course_id │ student_name    │ enrollment_date │   │
 * │   ├────────────┼───────────┼─────────────────┼─────────────────┤   │
 * │   │ 1          │ 101       │ Asha Sharma     │ 2026-01-01      │   │
 * │   │ 1          │ 102       │ Asha Sharma     │ 2026-01-05      │   │
 * │   │ 2          │ 101       │ Ravi Kumar      │ 2026-01-02      │   │
 * │   └────────────┴───────────┴─────────────────┴─────────────────┘   │
 * │                                                                    │
 * │   PROBLEM: student_name depends ONLY on student_id,                │
 * │            NOT on the full (student_id, course_id) key.            │
 * │                                                                    │
 * │   This is a PARTIAL DEPENDENCY.                                    │
 * │   It violates 2NF.                                                 │
 * │                                                                    │
 * │   enrollment_date depends on the full key (student + course)       │
 * │   That's fine.                                                     │
 * │                                                                    │
 * │   student_name depends only on student_id → PARTIAL → BAD          │
 * │                                                                    │
 * │ CORRECT DESIGN (2NF compliant):                                    │
 * │                                                                    │
 * │   students table:                                                  │
 * │   ┌────────────┬───────────────┐                                   │
 * │   │ student_id │ student_name  │                                   │
 * │   ├────────────┼───────────────┤                                   │
 * │   │ 1          │ Asha Sharma   │                                   │
 * │   │ 2          │ Ravi Kumar    │                                   │
 * │   └────────────┴───────────────┘                                   │
 * │                                                                    │
 * │   enrollments table:                                               │
 * │   ┌────────────┬───────────┬─────────────────┐                     │
 * │   │ student_id │ course_id │ enrollment_date │                     │
 * │   ├────────────┼───────────┼─────────────────┤                     │
 * │   │ 1          │ 101       │ 2026-01-01      │                     │
 * │   │ 1          │ 102       │ 2026-01-05      │                     │
 * │   │ 2          │ 101       │ 2026-01-02      │                     │
 * │   └────────────┴───────────┴─────────────────┘                     │
 * │                                                                    │
 * │   Now:                                                             │
 * │     → student_name depends ONLY on student_id (in students table)  │
 * │     → enrollment_date depends on FULL key (in enrollments)         │
 * │     → No partial dependencies                                      │
 * │                                                                    │
 * │ RULE: Every non-key column must depend on the WHOLE key,           │
 * │       not part of it.                                              │
 * │                                                                    │
 * │ If your table has a SINGLE primary key:                            │
 * │   It automatically satisfies 2NF (if 1NF is satisfied).            │
 * └────────────────────────────────────────────────────────────────────┘
 */

In [ ]:
/**
 * ─────────────────────────────────────────────────────────────────────
 * 7. THIRD NORMAL FORM (3NF) — No transitive dependencies
 * ─────────────────────────────────────────────────────────────────────
 *
 * ┌────────────────────────────────────────────────────────────────────┐
 * │ THIRD NORMAL FORM (3NF) removes TRANSITIVE DEPENDENCIES.           │
 * │                                                                    │
 * │ A table is in 3NF if:                                              │
 * │   → It is already in 2NF                                           │
 * │   → Non-key columns do NOT depend on other non-key columns         │
 * │                                                                    │
 * │ WHAT IS TRANSITIVE DEPENDENCY?                                     │
 * │                                                                    │
 * │   BAD DESIGN (violates 3NF):                                       │
 * │                                                                    │
 * │   enrollments table                                                │
 * │   PRIMARY KEY = (student_id, course_id)                            │
 * │                                                                    │
 * │   ┌────────────┬───────────┬────────────┬─────────────────┐        │
 * │   │ student_id │ course_id │ payment_id │ payment_amount  │        │
 * │   ├────────────┼───────────┼────────────┼─────────────────┤        │
 * │   │ 1          │ 101       │ pay_1001   │ 499.00          │        │
 * │   │ 1          │ 102       │ pay_1002   │ 799.00          │        │
 * │   └────────────┴───────────┴────────────┴─────────────────┘        │
 * │                                                                    │
 * │   DEPENDENCY CHAIN:                                                │
 * │     (student_id, course_id) → payment_id → payment_amount          │
 * │                                                                    │
 * │   payment_amount depends on payment_id,                            │
 * │   NOT directly on the primary key.                                 │
 * │                                                                    │
 * │   This is a TRANSITIVE DEPENDENCY.                                 │
 * │   It violates 3NF.                                                 │
 * │                                                                    │
 * │ CORRECT DESIGN (3NF compliant):                                    │
 * │                                                                    │
 * │   payments table:                                                  │
 * │   ┌────────────┬─────────────────┬────────────┐                    │
 * │   │ payment_id │ enrollment_id   │ amount     │                    │
 * │   ├────────────┼─────────────────┼────────────┤                    │
 * │   │ pay_1001   │ 1               │ 499.00     │                    │
 * │   │ pay_1002   │ 2               │ 799.00     │                    │
 * │   └────────────┴─────────────────┴────────────┘                    │
 * │                                                                    │
 * │   enrollments table:                                               │
 * │   ┌────────────┬───────────┬────────────┐                          │
 * │   │ enrollment │ student_id│ course_id  │                          │
 * │   │ _id        │           │            │                          │
 * │   ├────────────┼───────────┼────────────┤                          │
 * │   │ 1          │ 1         │ 101        │                          │
 * │   │ 2          │ 1         │ 102        │                          │
 * │   └────────────┴───────────┴────────────┘                          │
 * │                                                                    │
 * │   Now:                                                             │
 * │     → amount depends ONLY on payment_id (primary key)              │
 * │     → payment_id references enrollment                             │
 * │     → No transitive dependencies                                   │
 * │                                                                    │
 * │ RULE: Non-key columns must depend ONLY on the key,                 │
 * │       nothing else.                                                │
 * │                                                                    │
 * │ 3NF IS THE STANDARD for most production databases.                 │
 * │ It eliminates most update, insert, and delete anomalies.           │
 * └────────────────────────────────────────────────────────────────────┘
 */